# ALPAR data flow

Combines `Data.md` (ALPAR command inputs) and `0_description_of_data.md` (local `data/` folders). `data/` is gitignored.

1. Local datasets
2. ALPAR pipeline and inputs
3. Bridging datasets to ALPAR
4. Inconsistencies found between the two docs

## 1. Local datasets

| Folder | Role | Usable as `alpar` input? |
|---|---|---|
| `data/cabbage/` | CABBAGE / EMBL-EBI AMR Portal parquet tables (phenotypes + in silico AMRFinderPlus genotypes) | No (long-format labels, gene calls, no sequences) |
| `data/camda_amr_challenge/` | CAMDA 2026 assembled genomes, 6 species-drug pairs | Yes for `-i` (after Resistant/Susceptible + `.fna` rename, already applied); still needs `--reference` |
| `data/card/` | Full CARD download: ARO ontology, models, FASTA, SNPs, CARD:Epi | No (manual reference only) |

### CABBAGE

| File | Rows | Cols | Row = |
|---|---|---|---|
| `phenotype.parquet` | 1,714,486 | 34 | isolate x antibiotic AST result |
| `genotype.parquet` | 1,370,205 | 33 | isolate x predicted AMR gene/element |
| `phenotype_genotype_merged.parquet` | 109,338 | 55 | phenotype row joined to genotype row |

```
phenotype.parquet     genotype.parquet
(BioSample_ID, assembly_ID, antibiotic_ontology)
        \               /
         \             /
   phenotype_genotype_merged.parquet  (inner join on the 3 keys)
```

- `BioSample_ID` (e.g. `SAMEA1028830`), `assembly_ID` (`GCA_...` / `ERZ...`), `antibiotic_ontology` (ARO-style drug ID).
- Merge is much smaller: kept only when sample + assembly + antibiotic all have a genotype call. One isolate repeats across drugs/genes.
- Useful phenotype columns: `SRA_accession`, `collection_year`, `country`, `host`, `ast_standard`, `laboratory_typing_method`, `measurement*`.
- Useful genotype columns: `gene_symbol`, `amr_element_symbol`, `class`, `subclass`, `region_start/end`, `amrfinderplus_method`, `reference_accession`, `reference_sequence_coverage/identity`.

Why not ALPAR input: `phenotype.parquet` is long with text labels (ALPAR wants wide 0/1/2); genotype is gene presence from AMRFinderPlus, not Snippy SNPs (closest to the gene presence/absence step); no genome sequences, only accessions.

### CAMDA 2026 AMR challenge

Training isolates come from CABBAGE; testing isolates are held out with no public labels. This copy is renamed for ALPAR (`Resistant`/`Susceptible`, `.fna`, decompressed).

```
camda_amr_challenge/
  README.txt
  training_dataset/<species>__<antibiotic>/{Resistant,Susceptible}/*.fna
  testing_dataset/<species>__<antibiotic>/*.fna      # unlabeled
```

| Pair | Train R | Train S | Test |
|---|---|---|---|
| acinetobacter_baumannii__imipenem | 400 | 400 | 250 |
| campylobacter_coli__tetracycline | 400 | 400 | 250 |
| campylobacter_jejuni__nalidixic_acid | 400 | 400 | 250 |
| escherichia_coli__piperacillin | 400 | 400 | 250 |
| klebsiella_pneumoniae__imipenem | 400 | 400 | 250 |
| streptococcus_pneumoniae__penicillin | 400 | 400 | 250 |
| **Total** | **2400** | **2400** | **1500** |

- Training: `alpar automatix -i` / `create_binary_tables -i` on `training_dataset/`. Testing: only `alpar prediction -i`.
- No ID table links train and test; no phenotype TSV is shipped (ALPAR builds it from folders).
- Use agreement: acknowledge CABBAGE/CAMDA, submit results to CAMDA proceedings; don't redistribute genomes without checking terms.

### CARD

Reference catalog (genes, mutations, ontology), not per-isolate AST. Commercial-use restriction (`CARD-Download-README.txt`); ontologies CC-BY 4.0.

```
aro.tsv/.json/.obo/.owl   <- ARO terms
  |- aro_index.tsv            detection model + GenBank accession + ARO
  |- aro_categories.tsv       family / drug class / mechanism vocabulary
  |- aro_categories_index.tsv GenBank accession x categories
  |- PMID.tsv                 literature per ARO term
  |- snps.txt                 curated mutations on variant models
  |- card.json                full RGI models (sequences, cutoffs, SNPs)
  |- nucleotide_*.fasta / protein_*.fasta   by model type
mo / ro / ncbi_taxonomy / viro, shortname_antibiotics, shortname_pathogens
card_epi_relationship.tsv -> card_epi_relationship_pmid.tsv  (join on re_id)
```

Join keys: `ARO Accession`, `CVTERM ID`, `CARD Short Name`, Protein/DNA accession (also in FASTA headers), `re_id`.

FASTA by model type: homolog (presence = resistance), variant (wild-type refs, need SNP screening or false positives), knockout, overexpression, rRNA gene variant (nucleotide only).

Not used by ALPAR: nothing in `sr_amr/*.py` reads `card.json`, ontologies or CARD:Epi. `sr_amr/card_data/` (`snps.txt`, `shortname_*.tsv`; added in commit 4c5f97c, "annotation function_added") is bundled/packaged but never read; treat as unfinished scaffolding. CABBAGE `antibiotic_ontology` is meant to line up with CARD ARO accessions.

## 2. ALPAR pipeline and inputs

`automatix` chains the stages; other subcommands run them one at a time.

```
genomes + reference
   v
create_binary_tables -> mutation TSV, phenotype TSV, strains.txt, random_names.txt, annotations
   |- binary_table_threshold (optional)
   |- phylogenetic_tree / panacota -> tree
   |- gwas  (mutation + phenotype + tree)
   |- prps  (mutation + tree)
   '- ml    (mutation + phenotype [+ PRPS] [+ strains.txt])
        |- structman  (feature importance + annotations + reference)
        '- prediction (new genomes or table + saved model)
```

### Genome folder (`-i`)

```
input_folder/<antibiotic_name>/{Resistant,Susceptible}/*.fna
```

- Folder name = antibiotic; must match `-a` / phenotype column.
- Case-sensitive `Resistant`/`Susceptible`; accepted extensions `.fna`, `.fasta`, `.faa`.
- Coding: Resistant = 1, Susceptible = 0, not listed = 2.
- `create_binary_tables` also accepts a text file of genome paths (no labels).

### Other inputs

- `--reference`: `.gbk`/`.gbff`; required for `automatix`, `create_binary_tables`, `structman`; optional for `prediction`.
- `strains.txt`: genome paths; used by `panacota`, `phylogenetic_tree`, `ml --sail`.
- `random_names.txt`: original ID -> random ID; `--random_names_dict`.
- `phenotype_table.tsv`: wide TSV, strains x antibiotics, 1/0/2. Example: `tool_results/CAMDA2025/phenotype_Neisseria_gonorrhoeae.tsv`.
- `binary_mutation_table.tsv`, `..._with_gene_presence_absence.tsv`, `..._threshold_*_percent.tsv`: strains x features, 0/1.

### Subcommands

| Command | Key inputs |
|---|---|
| `automatix` | `-i` folder, `-o`, `--reference`; opt. `--prokka_custom_database`, `--bakta_db` |
| `create_binary_tables` | `-i` folder or path list, `-o`, `--reference`; opt. `--create_phenotype_from_folder` |
| `create_phenotype_table` | `-i` folder, `-o`; opt. `--random_names_dict` |
| `binary_table_threshold` | `-i` mutation TSV, `-o`; `--threshold_percentage` (default 0.2) |
| `phylogenetic_tree` | `-i` strains.txt/folder (Mashtree -> `phylogenetic_tree.tree`) |
| `panacota` | `-i` strains.txt/folder; `--data_type nucl\|prot` (-> `phylogenetic_tree.newick`) |
| `gwas` | `-i` mutation TSV, `-p` phenotype, `-t` tree, `-o` |
| `prps` | `-i` mutation TSV, `-t` tree, `-o` |
| `ml` | `-i`, `-p`, `-a` (one antibiotic), `-o`; opt. `--annotation`, `--prps`, `--prps_percentage` (30), `--sail`, `--sail_distance_matrix`, explicit train/test/validation lists; `--save_model`, `--feature_importance_analysis` |
| `structman` | `-i` feature-importance file(s), `--annotation`, `--reference`, `-o` |
| `prediction` | `-b` training mutation TSV, `--model`, `--model_names`, `-o`, and `-i` genomes or `-p` table; `--reference` if `-i`; opt. `--prps PATH PCT` |

`ml --ml_algorithm`: `rf` (default), `svm`, `gb`, `histgb`, `xgb`, `lr`.

### Which file goes where

| You have | Pass to |
|---|---|
| Genome folder | `automatix -i`, `create_binary_tables -i`, `create_phenotype_table -i`, optionally tree/sail/prediction |
| FASTA path list | `create_binary_tables -i`, `panacota -i`, `phylogenetic_tree -i`, `ml --sail`, `prediction -i` |
| Reference gbk/gbff | `--reference` of automatix, create_binary_tables, structman, prediction |
| `binary_mutation_table*.tsv` | `binary_table_threshold`, `gwas`, `prps`, `ml -i`, `prediction -b` |
| `phenotype_table.tsv` | `gwas -p`, `ml -p` |
| Tree | `gwas -t`, `prps -t` |
| PRPS scores | `ml --prps`, `prediction --prps` |
| `mutations_annotations.tsv` | `ml --annotation`, `structman --annotation` |
| Feature-importance file | `structman -i` |
| Saved model | `prediction --model` |

## 3. Bridging datasets to ALPAR

- **CAMDA training** (ready after rename): each `<species>__<antibiotic>` is one ALPAR "antibiotic" folder, so the resulting phenotype column / `-a` value is the full `species__antibiotic` string. A single `-i training_dataset/` run mixes six species with one reference, which is not meaningful; run per pair (pass the pair's parent layout, e.g. a folder containing just that one pair) with a species-appropriate `--reference`.
- **CAMDA testing**: `alpar prediction -i` with a saved model trained on the matching pair.
- **CABBAGE**: only for labels/metadata. Pivot to wide 0/1/2, download assemblies by `assembly_ID` from NCBI/ENA, then arrange as Resistant/Susceptible folders.
- **CARD**: manual cross-referencing of gene names/mutations ALPAR reports (ARO, `snps.txt`, short names).

## 4. Inconsistencies found

| # | Where | Issue | Evidence / resolution |
|---|---|---|---|
| 1 | `Data.md` CAMDA section vs `0_description_of_data.md` | **Direct contradiction.** `Data.md` says folders are lowercase `resistant`/`susceptible`, files are `.fa`/`.fa.gz`, testing is gzipped, and `zip_source_files/` exists, so it is "not usable as-is". `0_description` says the copy was renamed to `Resistant`/`Susceptible` and `.fna`. | Disk check: `training_dataset/acinetobacter_baumannii__imipenem/` contains `Resistant`, `Susceptible`. `0_description` is current; `Data.md` section is stale. Notebook follows the renamed layout. |
| 2 | `Data.md` CAMDA vs ALPAR rules | `Data.md` says the CAMDA layout is "species-antibiotic -> resistant/susceptible" matching ALPAR, but ALPAR treats the top folder as the antibiotic name, which must match `-a`. The `species__antibiotic` name becomes the column name, and species are mixed under one reference. | Neither doc mentions this; added in section 3. |
| 3 | `Data.md` extension note | States accepted extensions `.fna`, `.fasta`, `.faa` but also that phenotype creation looks for `.fna` only (and `0_description` says `.fna`). | `sr_amr/amr.py` lines 632 and 1705 accept all three for reading; confirm the phenotype step before relying on `.fasta`/`.faa`. Using `.fna` is safe. |
| 4 | `Data.md` automatix switches vs `ml` | `automatix` lists `--ml_algorithm rf svm lr xgb`, while `ml` lists `rf, svm, gb, histgb, xgb, lr`. | `amr.py:24` accepts all six; the automatix list is just incomplete. |
| 5 | CABBAGE join keys | `0_description` says merge joins on `BioSample_ID` + `assembly_ID` + `antibiotic_ontology`; `Data.md` says "assembly/antibiotic". | Minor wording; three keys is the fuller statement. |
| 6 | `phenotype.parquet` labels | `Data.md`: `resistant/susceptible/etc.`; `0_description` only shows those two. Other values (e.g. intermediate) may exist. | Check unique values before encoding to 0/1/2 (cell below). |
| 7 | `sr_amr/card_data/` | `0_description` says ALPAR "ships a small CARD subset" without saying it is unused; `Data.md` says it is never read. | Not a conflict; `Data.md` has the fuller statement. |
| 8 | `0_description` CAMDA | Says `-i` accepts `training_dataset/` directly; ignores that testing folders lack R/S split and that `README.txt` sits at the top level. | Harmless; testing must not be passed to `-i` for training. |
| 9 | Short-name example | `Data.md` cites `Eco_gyrA_FQ`, `0_description` cites `Abau_gyrA_FLO`; both follow the Species_gene_drug pattern. | Consistent. |

Numeric facts agree across both docs: CABBAGE row counts, 800 train (400/400) + 250 test per pair, 6 pairs = 2400 + 2400 train and 1500 test.

## Optional checks (run from repo root)

In [ ]:
from pathlib import Path

root = Path('data/camda_amr_challenge')
for pair in sorted((root / 'training_dataset').iterdir()):
    if not pair.is_dir() or pair.name == 'zip_source_files':
        continue
    r = len(list((pair / 'Resistant').glob('*.fna')))
    s = len(list((pair / 'Susceptible').glob('*.fna')))
    t = len(list((root / 'testing_dataset' / pair.name).glob('*.fna')))
    print(f'{pair.name:45s} R={r:4d} S={s:4d} test={t:4d}')

In [ ]:
import pandas as pd

pheno = pd.read_parquet('data/cabbage/phenotype.parquet', columns=['resistance_phenotype'])
print(pheno['resistance_phenotype'].value_counts(dropna=False))